## Open RayCloudTools data 

Work with QSMs from **RayCloudTools/RayExtract** (files) to extract tree-level metrics. #15Oct2026

**Ankasa plot** (50 x 40 m2)  Ghana

Actual plot location is missing

No L2A (FH) data available --- being processed by the ESA Team 

# Tree file. Optional per-tree attributes (e.g. 'height, crown_radius, ') followed by 'x,y,z,radius' and any additional per-segment attributes:
# x,y,z,radius,parent_id,section_id
533919.4054,582308.7427,123.5200,0.0540,-1,0, 
533919.4160,582308.7657,124.2056,0.0540,0,2225, 
533919.4044,582308.7294,124.4730,0.0540,1,4450, 
533919.4156,582308.7205,124.7852,0.0540,2,6916, 
533919.4304,582308.7174,125.0167,0.0540,3,9473, 
533919.4477,582308.7157,125.2863,0.0540,4,12239, 
533919.4716,582308.7066,125.5426,0.0540,5,15178, ... 533919.4600,582307.7500,128.5312,0.0168,45,138742 # last segment

In [ ]:
# Open a RayCloudTools output - Tree-level metrics at plot (QSMs)
file_path = '/home/jovyan/Examples_and_Data/Workshops/BIOMASS-Coding2026/TLS/Ankasa/c01_r02_subsampled_1cm_trees.txt'

In [ ]:
## Dataframe ::: DBH , Ht , Tree Volume ,
import numpy as np
import pandas as pd

# List to hold the row data
data_records = []

# Open the text file and skip the first 2 rows
with open(file_path, "r") as file:
    lines = file.readlines()[2:]

# Loop through every single row in the file
for line_num, line in enumerate(lines, start=3):
    if not line.strip():
        continue

    # Split the row by commas and convert to numbers
    tokens = [float(x) for x in line.strip().split(",") if x.strip()]

    # Reset variables specifically for THIS row/tree
    row_volume = 0.0
    prev_x, prev_y, prev_z = None, None, None
    
    # Trackers for height and DBH
    first_z = None
    last_z = None
    dbh_qsm = None  # Will hold the diameter once found

    # Step through the current row 6 values at a time
    for i in range(0, len(tokens), 6):
        chunk = tokens[i : i + 6]
        if len(chunk) < 6:
            continue
        x, y, z, radius, parent_id, section_id = chunk

        # Track Z values for height calculation
        if first_z is None:
            first_z = z  # Ground level reference
        last_z = z

        # Extract DBH: Find the first cylinder at or above 1.25m from ground
        if dbh_qsm is None and (z - first_z) >= 1.25:
            dbh_qsm = radius * 2

        # If it's the root node, establish the starting position
        if parent_id == -1:
            prev_x, prev_y, prev_z = x, y, z
            continue

        # Calculate the cylinder segment if we have a starting point
        if prev_x is not None:
            length = np.sqrt(
                (x - prev_x) ** 2 + (y - prev_y) ** 2 + (z - prev_z) ** 2
            )
            cylinder_volume = np.pi * (radius**2) * length
            row_volume += cylinder_volume

            # Move the starting point forward to the current node
            prev_x, prev_y, prev_z = x, y, z

    # Calculate tree height
    tree_height = (last_z - first_z) if first_z is not None else 0.0
    
    # Fallback if a tree is shorter than 1.25m and no DBH cylinder was reached
    if dbh_qsm is None:
        dbh_qsm = 0.0 #Implemented

    # Append all metadata to your records list
    data_records.append({
        "RowNumber": line_num, 
        "TotalVolume": row_volume,
        "TreeHeight": tree_height,
        "DBHqsm": dbh_qsm
    })

In [ ]:
# Create the final DataFrame
df = pd.DataFrame(data_records)

# Display the updated DataFrame
print("\nGenerated DataFrame with DBHqsm:")
print(df)

In [ ]:
## Dataframe properties

# 1. Check the dimensions
print(f"DataFrame Shape: {df.shape}")
print(f"Total Rows: {df.shape[0]}")
print(f"Maximum values found in a single row: {df.shape[1]}\n")

# 2. View the distribution of elements per row
# This tells you how long each individual row originally was
values_per_row = df.notna().sum(axis=1)
print("--- Distribution of Values per Row ---")
print(values_per_row.describe())

# 3. Print the technical overview
print("\n--- DataFrame Structural Info ---")
df.info()

# 4. Preview the actual data matrix
print("\n--- First 5 Rows Preview ---")
print(df.head())


In [ ]:
# Convert DBHqsm from meters to centimeters
df["DBHqsm"] = df["DBHqsm"] * 100

# Optional: Rename the column to reflect the new units
df.rename(columns={"DBHqsm": "DBHqsm_cm"}, inplace=True)

# Generate summary statistics for just Volume and Height
summary_stats = df[["TotalVolume", "TreeHeight", "DBHqsm_cm"]].describe()

# Display the summary table
print("\n=== Summary Statistics ===")
print(summary_stats.round(4))  # .round(4) keeps the output clean

In [ ]:
# Calculate the grand total volume
grand_total_volume = df["TotalVolume"].sum()

total_agbiomass = df["TotalVolume"].sum() * 0.5 # Wood density - Biomass convertion
 
print(f"\nGrand Total Volume for All Rows: {grand_total_volume:.4f} m3")

print(f"\nBiomass in plot: {total_agbiomass:.4f} Kg")

In [ ]:
# Visualize Ht and Volume distribution
import matplotlib.pyplot as plt
import seaborn as sns

# Set a clean visual style
sns.set_theme(style="whitegrid")

# Create a figure with 2 subplots side-by-side
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# 1. Distribution Plot for Total Volume
sns.histplot(
    data=df,
    x="TotalVolume",
    kde=True,
    color="forestgreen",
    ax=axes[0],
    bins=10,
)
axes[0].set_title("Distribution of Tree Volume", fontsize=14, pad=10)
axes[0].set_xlabel("Total Volume (cubic units)", fontsize=12)
axes[0].set_ylabel("Count", fontsize=12)

# 2. Distribution Plot for Tree Height
sns.histplot(
    data=df, 
    x="TreeHeight", 
    kde=True, 
    color="darkblue", 
    ax=axes[1], 
    bins=10
)
axes[1].set_title("Distribution of Tree Height", fontsize=14, pad=10)
axes[1].set_xlabel("Tree Height (units)", fontsize=12)
axes[1].set_ylabel("Count", fontsize=12)

# Optimize layout and display the plots
plt.tight_layout()
plt.show()


In [ ]:
# Subset DataFrame - TreeVolume Range (10-300)
df_subset = df[df["TotalVolume"].between(12, 4000)].copy()

# View the new filtered DataFrame
print(f"Subset created with {len(df_subset)} rows.")
print(df_subset)

In [ ]:
# 1. Generate summary statistics for the 0-100 subset
print("\n=== Summary Statistics (Volume 0 to 100) ===")
print(df_subset.describe().round(4))

# 2. Plot the height vs volume for this specific subset
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.figure(figsize=(7, 5))

sns.histplot(data=df_subset, x="TotalVolume", kde=True, color="teal", bins=8)

plt.title("Distribution of Volume (Filtered: 0 to 100)")
plt.xlabel("Total Volume")
plt.ylabel("Count")
plt.xlim(0, 120)

plt.show()

In [ ]:
# A subset for rows with TotalVolume greater than ...
vol_threshold = 30

df_above_xx = df[df["TotalVolume"] > vol_threshold].copy()

# Display how many rows match the criteria
print(f"Subset created with {len(df_above_xx)} rows out of {len(df)} total rows. \n")
print(df_above_xx)

**The End** ________